In [1]:
import subprocess, sys

packages = ["onnx", "onnxruntime", "transformers", "torch"]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        version_line = [l for l in result.stdout.split("\n") if l.startswith("Version")][0]
        print(f"✅ {pkg}: {version_line}")
    else:
        print(f"❌ {pkg}: NOT INSTALLED")

✅ onnx: Version: 1.21.0
✅ onnxruntime: Version: 1.24.4
✅ transformers: Version: 5.4.0
✅ torch: Version: 2.11.0


In [2]:
import torch
import time
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# 加载模型和 tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

# 准备输入
prompt = "The quick brown fox jumps over the lazy dog"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]

# 测量生成速度（生成 50 个新 token）
NEW_TOKENS = 50
RUNS = 3

times = []
with torch.no_grad():
    for i in range(RUNS):
        start = time.perf_counter()
        output = model.generate(
            input_ids,
            max_new_tokens=NEW_TOKENS,
            do_sample=False,          # greedy，结果确定性
            pad_token_id=tokenizer.eos_token_id,
        )
        end = time.perf_counter()
        times.append(end - start)

avg_time = sum(times) / RUNS
tokens_per_sec = NEW_TOKENS / avg_time

# 解码输出
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print(f"输入 token 数: {input_ids.shape[1]}")
print(f"生成 token 数: {NEW_TOKENS}")
print(f"平均耗时: {avg_time:.3f}s ({RUNS} 次平均)")
print(f"生成速度: {tokens_per_sec:.1f} tokens/sec")
print(f"\n生成文本:\n{generated_text}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


输入 token 数: 9
生成 token 数: 50
平均耗时: 1.138s (3 次平均)
生成速度: 43.9 tokens/sec

生成文本:
The quick brown fox jumps over the lazy dog and runs off.

"I'm sorry, I'm sorry, I'm sorry," the fox says.

"I'm sorry, I'm sorry, I'm sorry," the fox says.

"I'm sorry, I


In [3]:
import torch
import os
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

# 准备一个 dummy 输入用于 tracing
dummy_input_ids = torch.randint(0, tokenizer.vocab_size, (1, 10))  # batch=1, seq_len=10

# 导出路径
save_dir = os.path.expanduser("~/ml-projects/quantization-exp")
os.makedirs(save_dir, exist_ok=True)
onnx_path = os.path.join(save_dir, "gpt2_no_cache.onnx")

# 导出
# use_cache=False：关闭 KV cache，让计算图简单干净
with torch.no_grad():
    torch.onnx.export(
        model,
        (dummy_input_ids,),
        onnx_path,
        input_names=["input_ids"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch_size", 1: "seq_len"},  # 两个维度都动态
            "logits":    {0: "batch_size", 1: "seq_len"},
        },
        opset_version=14,
        do_constant_folding=True,
        export_params=True,
    )

# 验证文件
size_mb = os.path.getsize(onnx_path) / 1024 / 1024
print(f"✅ 导出成功: {onnx_path}")
print(f"文件大小: {size_mb:.1f} MB")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/var/folders/lv/mnjxv0d53_xczbffr49m5z_40000gn/T/ipykernel_33984/704389473.py:21: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0419 23:52:47.310000 33984 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `GPT2LMHeadModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GPT2LMHeadModel([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `GPT2LMHeadModel([...]` with `torch.export.export(..., strict=True)`...
[torch.onnx] Obtain model graph for `GPT2LMHeadModel([...]` with `torch.export.export(..., strict=True)`... ❌


TorchExportError: Failed to export the model with torch.export. [96mThis is step 1/3[0m of exporting the model to ONNX. Next steps:
- Modify the model code for `torch.export.export` to succeed. Refer to https://pytorch.org/docs/stable/generated/exportdb/index.html for more information.
- Debug `torch.export.export` and submit a PR to PyTorch.
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.

## Exception summary

<class 'RuntimeError'>: Found <class 'transformers.cache_utils.DynamicCache'> in output, which is not a known type. If this type holds tensors, you need to register a pytree for it. See https://github.com/pytorch/functorch/issues/475 for a brief explanation why. If you don't need to register a pytree, please leave a comment explaining your use case and we'll make this more ergonomic to deal with

(Refer to the full stack trace above for more information.)

In [4]:
import torch.nn as nn
from transformers import GPT2LMHeadModel

# 验证继承关系
print(isinstance(GPT2LMHeadModel.from_pretrained("gpt2"), nn.Module))  # True

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

True


In [5]:
import torch
import os
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
base_model = GPT2LMHeadModel.from_pretrained(model_name)
base_model.eval()

# Wrapper：只返回 logits，屏蔽 DynamicCache
class GPT2LogitsOnly(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        # use_cache=False：不生成 past_key_values，输出干净
        out = self.model(input_ids, use_cache=False)
        return out.logits  # 只返回 logits tensor

wrapped = GPT2LogitsOnly(base_model)
wrapped.eval()

# dummy 输入
dummy_input_ids = torch.randint(0, tokenizer.vocab_size, (1, 10))

# 验证 wrapper 输出类型
with torch.no_grad():
    test_out = wrapped(dummy_input_ids)
print(f"wrapper 输出类型: {type(test_out)}")
print(f"wrapper 输出 shape: {test_out.shape}")  # 期望 (1, 10, 50257)

# 导出
save_dir = os.path.expanduser("~/ml-projects/quantization-exp")
os.makedirs(save_dir, exist_ok=True)
onnx_path = os.path.join(save_dir, "gpt2_no_cache.onnx")

with torch.no_grad():
    torch.onnx.export(
        wrapped,
        (dummy_input_ids,),
        onnx_path,
        input_names=["input_ids"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch_size", 1: "seq_len"},
            "logits":    {0: "batch_size", 1: "seq_len"},
        },
        opset_version=14,
        do_constant_folding=True,
        export_params=True,
    )

size_mb = os.path.getsize(onnx_path) / 1024 / 1024
print(f"\n✅ 导出成功: {onnx_path}")
print(f"文件大小: {size_mb:.1f} MB")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/var/folders/lv/mnjxv0d53_xczbffr49m5z_40000gn/T/ipykernel_33984/953436198.py:39: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0419 23:57:24.103000 33984 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


wrapper 输出类型: <class 'torch.Tensor'>
wrapper 输出 shape: torch.Size([1, 10, 50257])
[torch.onnx] Obtain model graph for `GPT2LogitsOnly([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `GPT2LogitsOnly([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/opt/anaconda3/envs/mlenv/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 14).
Failed to convert the model to the target version 14 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/opt/anaconda3/envs/mlenv/lib/python3.11/site-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/mlenv/lib/python3.11/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/opt/anaconda3/envs/mlenv/lib/python3.11/site

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
Applied 127 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅

✅ 导出成功: /Users/tongwenchao/ml-projects/quantization-exp/gpt2_no_cache.onnx
文件大小: 1.3 MB


In [6]:
import os
import glob

save_dir = os.path.expanduser("~/ml-projects/quantization-exp")

files = sorted(glob.glob(os.path.join(save_dir, "gpt2*")))
for f in files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f"{os.path.basename(f):40s}  {size_mb:.1f} MB")

gpt2_no_cache.onnx                        1.3 MB
gpt2_no_cache.onnx.data                   474.7 MB


In [7]:
import torch
import numpy as np
import onnxruntime as ort
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import os

# 加载 PyTorch 模型（用于对比）
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
pt_model = GPT2LMHeadModel.from_pretrained(model_name)
pt_model.eval()

# 加载 ONNX 模型
onnx_path = os.path.expanduser("~/ml-projects/quantization-exp/gpt2_no_cache.onnx")
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

# 准备输入
prompt = "The quick brown fox"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids_np = inputs["input_ids"].numpy()  # onnxruntime 需要 numpy

# PyTorch 推理
with torch.no_grad():
    pt_logits = pt_model(inputs["input_ids"], use_cache=False).logits.numpy()

# ONNX 推理
ort_logits = sess.run(["logits"], {"input_ids": input_ids_np})[0]

# 对比输出
max_diff = np.abs(pt_logits - ort_logits).max()
mean_diff = np.abs(pt_logits - ort_logits).mean()

print(f"输入 shape: {input_ids_np.shape}")
print(f"输出 shape: PT={pt_logits.shape}, ORT={ort_logits.shape}")
print(f"最大绝对误差: {max_diff:.6f}")
print(f"平均绝对误差: {mean_diff:.6f}")
print(f"\n数值一致性: {'✅ 通过' if max_diff < 1e-3 else '❌ 差异过大'}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

输入 shape: (1, 4)
输出 shape: PT=(1, 4, 50257), ORT=(1, 4, 50257)
最大绝对误差: 0.000366
平均绝对误差: 0.000103

数值一致性: ✅ 通过


In [8]:
import torch
import numpy as np
import onnxruntime as ort
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import time, os

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
pt_model = GPT2LMHeadModel.from_pretrained(model_name)
pt_model.eval()

onnx_path = os.path.expanduser("~/ml-projects/quantization-exp/gpt2_no_cache.onnx")
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

prompt = "The quick brown fox jumps over the lazy dog"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids_pt = inputs["input_ids"]
input_ids_np = input_ids_pt.numpy()

NEW_TOKENS = 50
RUNS = 3

# ── PyTorch greedy 生成（用 .generate()）──
pt_times = []
with torch.no_grad():
    for _ in range(RUNS):
        start = time.perf_counter()
        out = pt_model.generate(
            input_ids_pt,
            max_new_tokens=NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        pt_times.append(time.perf_counter() - start)

pt_avg = sum(pt_times) / RUNS

# ── ONNX 手写 greedy 生成循环 ──
# （ONNX 没有 .generate()，必须手写循环，每步调一次 forward）
def onnx_generate(sess, input_ids_np, new_tokens):
    ids = input_ids_np.copy()
    for _ in range(new_tokens):
        logits = sess.run(["logits"], {"input_ids": ids})[0]  # (1, seq, vocab)
        next_token = logits[0, -1, :].argmax()                # greedy: 取最大 logit
        ids = np.concatenate([ids, [[next_token]]], axis=1)   # append 到序列
    return ids

ort_times = []
for _ in range(RUNS):
    start = time.perf_counter()
    onnx_generate(sess, input_ids_np, NEW_TOKENS)
    ort_times.append(time.perf_counter() - start)

ort_avg = sum(ort_times) / RUNS

# ── 结果 ──
print(f"{'':20s} {'平均耗时':>10s}  {'tokens/sec':>12s}")
print(f"{'PyTorch':20s} {pt_avg:>10.3f}s  {NEW_TOKENS/pt_avg:>10.1f} t/s")
print(f"{'ONNX (no cache)':20s} {ort_avg:>10.3f}s  {NEW_TOKENS/ort_avg:>10.1f} t/s")
print(f"\nONNX 相对 PyTorch: {pt_avg/ort_avg:.2f}x")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

                           平均耗时    tokens/sec
PyTorch                   0.908s        55.1 t/s
ONNX (no cache)           1.416s        35.3 t/s

ONNX 相对 PyTorch: 0.64x


In [9]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

# 在 forward 上打 patch，记录每次调用时的输入序列长度
call_log = []
original_forward = model.forward

def patched_forward(*args, **kwargs):
    input_ids = kwargs.get("input_ids", args[0] if args else None)
    past = kwargs.get("past_key_values", None)
    if input_ids is not None:
        call_log.append({
            "input_len": input_ids.shape[1],
            "has_past":  past is not None,
        })
    return original_forward(*args, **kwargs)

model.forward = patched_forward

prompt = "The quick brown fox"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    model.generate(
        inputs["input_ids"],
        max_new_tokens=5,       # 只生成 5 个 token，够看规律
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(f"{'step':>5}  {'input_len':>10}  {'has_past_kv':>12}")
for i, log in enumerate(call_log):
    print(f"{i:>5}  {log['input_len']:>10}  {str(log['has_past']):>12}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

 step   input_len   has_past_kv
    0           4          True
    1           1          True
    2           1          True
    3           1          True
    4           1          True


In [10]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

prompt = "The quick brown fox"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]  # shape: (1, 4)

with torch.no_grad():
    # step 0: prefill，past=None
    out0 = model(input_ids, use_cache=True)
    logits0 = out0.logits
    past = out0.past_key_values  # 12 层的 cache

    # step 1: 只传新 token
    next_token = logits0[0, -1, :].argmax().unsqueeze(0).unsqueeze(0)  # shape: (1,1)
    out1 = model(next_token, past_key_values=past, use_cache=True)

print(f"输入 prompt token 数: {input_ids.shape[1]}")
print(f"step 0 input shape:  {input_ids.shape}")
print(f"step 0 logits shape: {logits0.shape}")
print(f"\npast_key_values: {len(past)} 层")
print(f"每层 tuple 长度: {len(past[0])} (key, value)")
print(f"单个 cache tensor shape: {past[0][0].shape}")
print(f"  → (batch, n_heads, seq_len, head_dim)")

print(f"\nstep 1 input shape:  {next_token.shape}")
print(f"step 1 logits shape: {out1.logits.shape}")
print(f"step 1 新 cache seq_len: {out1.past_key_values[0][0].shape[2]}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

输入 prompt token 数: 4
step 0 input shape:  torch.Size([1, 4])
step 0 logits shape: torch.Size([1, 4, 50257])

past_key_values: 12 层


TypeError: 'DynamicCache' object is not subscriptable

In [11]:
print(f"past 类型: {type(past)}")
print(f"past 的方法/属性: {[x for x in dir(past) if not x.startswith('__')]}")

past 类型: <class 'transformers.cache_utils.DynamicCache'>
past 的方法/属性: ['batch_repeat_interleave', 'batch_select_indices', 'crop', 'early_initialization', 'get_mask_sizes', 'get_max_cache_shape', 'get_seq_length', 'is_compileable', 'is_initialized', 'is_sliding', 'layer_class_to_replicate', 'layers', 'max_batch_size', 'max_cache_len', 'offload', 'offloading', 'prefetch', 'reorder_cache', 'reset', 'update']


In [12]:
print(f"layers 类型: {type(past.layers)}")
print(f"layers 长度: {len(past.layers)}")
print()
layer0 = past.layers[0]
print(f"layer[0] 类型: {type(layer0)}")
print(f"layer[0] 的属性: {[x for x in dir(layer0) if not x.startswith('__')]}")

layers 类型: <class 'list'>
layers 长度: 12

layer[0] 类型: <class 'transformers.cache_utils.DynamicLayer'>
layer[0] 的属性: ['_abc_impl', 'batch_repeat_interleave', 'batch_select_indices', 'crop', 'device', 'dtype', 'get_mask_sizes', 'get_max_cache_shape', 'get_seq_length', 'is_compileable', 'is_initialized', 'is_sliding', 'keys', 'lazy_initialization', 'offload', 'prefetch', 'reorder_cache', 'reset', 'update', 'values']


In [13]:
layer0 = past.layers[0]
k = layer0.keys
v = layer0.values

print(f"keys 类型:  {type(k)}")
print(f"values 类型: {type(v)}")
print()
print(f"keys shape:   {k.shape}")
print(f"values shape: {v.shape}")
print(f"  → (batch, n_heads, seq_len, head_dim)")

keys 类型:  <class 'torch.Tensor'>
values 类型: <class 'torch.Tensor'>

keys shape:   torch.Size([1, 12, 5, 64])
values shape: torch.Size([1, 12, 5, 64])
  → (batch, n_heads, seq_len, head_dim)


In [14]:
print(f"out0 之后 cache seq_len: {past.layers[0].keys.shape[2]}")       # 应该是 5（out1 已更新）
print(f"out1 之后 cache seq_len: {out1.past_key_values.layers[0].keys.shape[2]}")  # 应该也是 5？还是 6？

out0 之后 cache seq_len: 5
out1 之后 cache seq_len: 5
